# 🩺 YaoBi-Skill 完美复现 · Colab + ngrok 公网 UI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/Yao-Bi-Agent/blob/claude/focused-planck-3dv9we/colab/YaoBi_Skill_Colab.ipynb)

> **沈钦荣腰痹经验规则智能体** — 一键在 Google Colab 复现**全部功能**，并通过 **ngrok** 把完整 UI 暴露为**公网链接**。

## 这个项目长什么样
- **前端 `frontend/`**：零依赖纯静态 SPA（`index.html` + `app.js` + `styles.css` + `mined_rules.js`）。总览看板、智能问诊（FSM）、智能问答、智能体协作、经验推理、经验总结、规则挖掘、证据回溯、医师审核、评估与安全、设置——共 **11 个模块**的逻辑全部在浏览器端自洽运行。**这就是要"完美复现"的 UI。**
- **后端 `backend/`**：纯 Python 技能库 + CLI（核心依赖仅 `pyyaml`）。规则引擎、11 个智能体共享黑板协作、xlsx 脱敏挖掘、可选 Tao（`CMLM/Dao1-30b-a3b`）大模型叠加。
- **没有独立 Web 服务**（`backend/api/` 仅是 FastAPI 占位）。所以"复现 UI" = 用 `http.server` 托管 `frontend/` 目录并经 **ngrok** 映射公网；前端不依赖后端 API，开箱即用、完整还原。

## 怎么用
**从上到下依次运行每个单元格。** 第 ③ 步会打印你的**公网 UI 链接**，点开即是完整界面；第 ④–⑨ 步逐一复现后端全部能力（CLI、多智能体、全量测试、挖掘、Tao）。

## ⚠️ 用途边界
仅用于名老中医经验研究、医案复盘、教学训练与医师复核。**不输出最终诊断、完整处方或患者可执行剂量**；附片、细辛、虫类药等均需执业医师审核签名。


## ① 克隆仓库 + 安装依赖

核心依赖仅 `pyyaml`；额外安装 `pyngrok`（公网映射）、`openpyxl` 与 `pytest`（挖掘与测试）。

In [ ]:
# === ① 克隆仓库 + 安装依赖 ===
import os, sys, subprocess

REPO_URL = "https://github.com/pariskang/Yao-Bi-Agent.git"
BRANCH   = "main"            # 如需特定分支可改，例如 "claude/focused-planck-3dv9we"
BASE     = "/content" if os.path.isdir("/content") else os.getcwd()
WORKDIR  = os.path.join(BASE, "Yao-Bi-Agent")

if not os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORKDIR], check=True)
else:
    print("仓库已存在，跳过克隆。")

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyyaml>=6.0", "pyngrok>=7.0", "openpyxl>=3.1", "pytest>=8.0"], check=True)
# 以可编辑模式安装本项目，使 `import backend...` 在任意目录可用
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("✅ 安装完成。")

## ② 配置 ngrok authtoken

免费注册 [ngrok](https://dashboard.ngrok.com/signup) 后，到 [Your Authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) 复制 token，运行下面单元格粘贴（输入不回显）。

> 没有 token？跳过本步，用第 ③-B 步的 Colab 内置代理也能打开 UI。

In [ ]:
# === ② 配置 ngrok authtoken（输入不会回显）===
# 获取地址：https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok
from getpass import getpass

_token = getpass("粘贴你的 ngrok authtoken：").strip()
if _token:
    ngrok.set_auth_token(_token)
    print("✅ ngrok authtoken 已配置。")
else:
    print("⚠️ 未输入 token。可改用第 ③-B 步的 Colab 内置代理（无需 token）。")

## ③ ★ 托管完整 UI 并映射公网（ngrok）

运行后会得到一个 `https://xxxx.ngrok-free.app` 链接——**这就是完整复现的公网 UI**。本单元格在后台线程托管 `frontend/`，保持内核运行即一直在线。

In [ ]:
# === ③ 托管完整 UI 并映射到公网 ===
import os, threading, functools, http.server, socketserver
from pyngrok import ngrok

PORT = 8000
FRONTEND_DIR = os.path.join(os.getcwd(), "frontend")
assert os.path.isfile(os.path.join(FRONTEND_DIR, "index.html")), \
    "未找到 frontend/index.html，请先运行第 ① 步克隆仓库。"

# 重复运行本单元格时，先清理旧服务与旧隧道
try:
    _httpd.shutdown(); _httpd.server_close()
except Exception:
    pass
try:
    ngrok.kill()
except Exception:
    pass

class _Server(socketserver.ThreadingTCPServer):
    allow_reuse_address = True
    daemon_threads = True

_handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=FRONTEND_DIR)
_httpd = _Server(("0.0.0.0", PORT), _handler)
threading.Thread(target=_httpd.serve_forever, daemon=True).start()
print(f"本地静态服务已启动：http://127.0.0.1:{PORT}  (目录 {FRONTEND_DIR})")

try:
    public_url = ngrok.connect(PORT).public_url
    print("=" * 64)
    print("✅  YaoBi-Skill 全功能 UI 已上线（公网可访问）")
    print("🔗  打开此链接：", public_url)
    print("=" * 64)
    print("提示：ngrok 免费版首次访问会出现确认页，点 \"Visit Site\" 即可进入。")
    print("本单元格保持后台运行；运行第 ⑨ 步或中断内核即可停止。")
except Exception as e:
    print("❌ ngrok 建立隧道失败：", e)
    print("   请确认已在第 ② 步配置 authtoken，或改用下方第 ③-B 步（Colab 内置代理，无需 token）。")

### ③-B 备选：无 ngrok token 时用 Colab 内置端口代理

已运行第 ③ 步启动本地服务后，运行下面单元格用 Colab 自带代理打开 UI（仅 Colab 内有效）。

In [ ]:
# === ③-B 备选：Colab 内置端口代理（无需 ngrok token）===
try:
    from google.colab.output import eval_js
    proxy = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print("🔗 Colab 代理链接：", proxy)
except Exception as e:
    print("非 Colab 环境或代理不可用：", e)

## ④ 后端规则管线 CLI（与 README 完全一致）

确定性规则引擎：结构化抽取 → 标签标准化 → 证型评分 → 方剂路线 → 药物模块 → 冲突/安全审查 → 报告。`TAO_BACKEND=mock` 让 `--use-llm` 演示 Tao 叠加层（无需 GPU）。

In [ ]:
# === ④ 后端规则管线 CLI ===
import os
os.environ["TAO_BACKEND"] = "mock"   # 让 Tao 相关演示用 mock 后端，无需 GPU

CASE = "患者女，68岁，腰痛反复5年，加重1月，伴下肢麻木，畏寒，舌暗苔白腻，脉细缓，既往骨质疏松。"

print(">>> 规则管线报告  (python -m backend.main --text ...)\n")
!python -m backend.main --text "{CASE}"

print("\n>>> 自主多步智能体  (--ask --autonomous)\n")
!python -m backend.main --ask "这个病人是什么证型、用什么方、有什么风险？" --autonomous

print("\n>>> Tao 直连对话（mock 后端）  (--tao-chat)\n")
!python -m backend.main --tao-chat "请用一句话解释本案规则线索"

print("\n>>> Tao 教学解释叠加层（mock 后端）  (--text --use-llm)\n")
!python -m backend.main --text "{CASE}" --use-llm

## ⑤ 多智能体 / 多轮问答 / 自动问诊（Python API）

直接调用后端：多轮智能问答（受限技能集内自主选 skill）、自主多步规划+子智能体委派、CaseGuide 有限状态机自动问诊 → 11 个智能体共享黑板协作 → 最终医案。

In [ ]:
# === ⑤ 多智能体 / 多轮问答 / 自动问诊 ===
import os
os.environ["TAO_BACKEND"] = "mock"

# (1) 多轮智能问答：语言模型在受限技能集内自主选择 skill，规则/挖掘数据作答
from backend.agents.conversation import ConversationSession
turn = ConversationSession(use_llm=False).ask("这个病人偏向什么证型？")
print(f"[问答] intent={turn['intent']} via={turn['method']} skills={turn['skills']}")
print(turn["answer"][:200], "...\n")

# (2) 自主多步智能体：规划 → 子智能体委派 → 综合（ReAct / Plan-and-Execute）
from backend.agents.autonomous_agent import AutonomousQAAgent
auto = AutonomousQAAgent(use_llm=False).run("这个病人是什么证型、用什么方、有什么风险？")
print("[自主计划]", " → ".join(p["label"] for p in auto["plan"]), f"(via={auto.get('plan_method')})\n")

# (3) CaseGuide 有限状态机：自动问诊 → 多智能体黑板协作 → 最终医案
from backend.skills.caseguide_state_machine import CaseGuideSession
s = CaseGuideSession()
answers = {
    "RF001": "否", "RF002": "否", "RF003": "否", "RF004": "否", "RF005": "否", "RF006": "否",
    "age": 68, "sex": "女", "main_symptom": "腰腿痛", "duration": "5年",
    "numbness": "经常有", "radiation": "到小腿", "cold_relation": "遇冷加重，热敷舒服",
    "tongue_color": "偏暗紫", "tongue_coating": "白腻",
}
script = s.run_scripted_interview(answers)
print(f"[FSM] 自动推进 {len(script['transcript'])} 个问诊状态")

collab = s.run_agent_collaboration()
print(f"[协作] {len(collab['collaboration_trace'])} 个智能体接力；红旗中止={collab['halted']}；Tao 在环={collab['llm_in_loop']}")
for a in collab["collaboration_trace"]:
    print(f"    {a['step']:>2}. {a['agent']:<22} [{a['kind']:>4}] {a['status']:<8} → {a['handoff_to']}")

report = s.final_report()
print(f"\n[最终医案] 质量评分={report['case_quality_score']} 等级={report['grade']}")
print("候选证型:", [c["name"] for c in report.get("syndrome_candidates", [])][:3])

## ⑥ 运行全量测试（证明后端全部功能可用）

共 **85** 项测试：红旗硬门控、Output Guard 禁用输出、挖掘脱敏、PII 泄漏检查、Tao 守卫与回退、前端静态资产等。

In [ ]:
# === ⑥ 全量测试 ===
!python -m pytest -q

## ⑦ xlsx 脱敏医案规则挖掘

挖掘管道把门诊导出转化为可审核的候选规则（频次 / 关联规则 support·confidence·lift / 签名方剂 / 剂量分布）。仓库已内置脱敏挖掘产物 `frontend/mined_rules.js`，UI 的「总览看板/规则挖掘/证据回溯」直接读取它。下面用内存中的脱敏 case 字典演示纯函数管道（不落盘、不覆盖已发布数据）。

In [ ]:
# === ⑦ xlsx 脱敏医案规则挖掘（纯函数演示）===
from backend.mining.xlsx_case_miner import mine_cases, build_rule_candidates

# 已脱敏的 case 字典（无任何身份信息，仅保留行号引用）
cases = [
    {"row_id": 2, "sex": "女", "age": 68, "age_band": "60s",
     "symptom_tags": ["lower_limb_numbness", "cold_aggravation", "elderly"],
     "zheng": "气血痹阻证", "tcm_diseases": ["腰痹"], "western_dx": ["腰椎间盘突出"],
     "herbs": ["独活", "桑寄生", "当归", "细辛", "牛膝"],
     "herb_doses": {"细辛": 3.0, "当归": 10.0}, "has_prescription": True, "osteoporosis_history": False},
    {"row_id": 3, "sex": "男", "age": 72, "age_band": "70s",
     "symptom_tags": ["lower_limb_numbness", "elderly", "osteoporosis"],
     "zheng": "肝肾不足证", "tcm_diseases": ["腰痹"], "western_dx": ["骨质疏松"],
     "herbs": ["独活", "桑寄生", "杜仲", "牛膝"],
     "herb_doses": {"杜仲": 12.0}, "has_prescription": True, "osteoporosis_history": True},
    {"row_id": 4, "sex": "女", "age": 61, "age_band": "60s",
     "symptom_tags": ["cold_aggravation", "elderly"],
     "zheng": "寒湿痹阻证", "tcm_diseases": ["腰痹"], "western_dx": ["腰肌劳损"],
     "herbs": ["独活", "桑寄生", "细辛", "桂枝"],
     "herb_doses": {"细辛": 3.0}, "has_prescription": True, "osteoporosis_history": False},
]

mined = mine_cases(cases)
stats = mined["dataset_stats"]
print("脱敏样本:", stats["n_cases"], "例，含处方", stats["n_with_prescription"], "例")
print("证型分布:", stats["zheng_distribution"])
print("高频药物:", dict(list(mined["herb_frequency_top"].items())[:6]))
print("签名方剂命中:", [h["formula"] for h in mined["formula_signature_hits"]])

candidates = build_rule_candidates(mined)
print(f"\n候选规则 {len(candidates)} 条（全部 pending_expert_review / clinician_only）:")
for r in candidates[:5]:
    print("  -", r["rule_id"], "|", r.get("rule_type"))

## ⑧ 可选：Tao（`CMLM/Dao1-30b-a3b`）大模型叠加

Tao 仅做教学解释/问诊改写，必经 JSON Repair + Output Guard，违规即回退确定性规则。下面 `mock` 后端随处可跑；真实 `transformers` 后端需加载 **30B** 模型，建议 A100/L4 等大显存 GPU。

In [ ]:
# === ⑧ Tao 大模型叠加 ===
# (A) mock 后端：随处可运行，演示安全管线与回退机制
from backend.llm.dao_client import DaoClient, DaoGenerationConfig
mock = DaoClient(DaoGenerationConfig(backend="mock"))
print("[mock] ", mock.chat([], "请解释本案规则线索"))

# (B) 真实本地推理（需大显存 GPU，30B 模型）——按需取消注释：
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "-q",
#                 "transformers>=4.44", "torch>=2.2", "accelerate>=0.33"], check=True)
# import os
# os.environ["TAO_BACKEND"] = "transformers"
# os.environ["TAO_MAX_NEW_TOKENS"] = "512"
# !python -m backend.main --tao-chat "请基于规则线索解释本案" --stream
print("\n（如需真实 30B 推理，取消上面 (B) 段注释，建议在 A100/L4 等大显存运行时执行。）")

## ⑨ 清理（停止公网隧道与本地服务）

In [ ]:
# === ⑨ 清理 ===
try:
    from pyngrok import ngrok
    ngrok.kill()
    print("已关闭 ngrok 隧道。")
except Exception as e:
    print("ngrok:", e)
try:
    _httpd.shutdown(); _httpd.server_close()
    print("已关闭本地 HTTP 服务。")
except Exception as e:
    print("httpd:", e)